In [ ]:
# colab 設定
# 学習時はcontent配下にDATA_ROOTを置く。
from google.colab import drive
drive.mount('/content/gdrive')

%mkdir /content/data
%mkdir /content/gdrive/MyDrive/深層学習スクラッチ/deeplearning_implementation/AlexNet/pth
DATA_ROOT = "content/data/tiny-imagenet-200"
PTH_ROOT = "/content/gdrive/MyDrive/深層学習スクラッチ/deeplearning_implementation/AlexNet/pth"
%cd /content/data 
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
import os
from pathlib import Path
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.transforms import v2



In [ ]:
def compute_rgb_statistics(loader):
    rgb_sum = torch.zeros(3, dtype=torch.float64)
    rgb_outer = torch.zeros((3, 3), dtype=torch.float64)
    n_pixels = 0

    for images, _ in loader:
        # images: [B, 3, H, W]
        pixels = images.permute(0, 2, 3, 1).reshape(-1, 3).double()
        rgb_sum += pixels.sum(dim=0)
        rgb_outer += pixels.T @ pixels
        n_pixels += pixels.shape[0]
    mean = rgb_sum / n_pixels
    covariance = rgb_outer / n_pixels - torch.outer(mean, mean)
    return mean.float(), covariance.float()

class MeanSubtraction:
    def __init__(self, mean):
        self.mean = torch.as_tensor(mean, dtype=torch.float32).view(3, 1, 1)

    def __call__(self, image):
        return image - self.mean.to(image.device)

class PCAColorAugmentation:
    def __init__(self, eigenvectors, eigenvalues, alpha_std=0.1):
        self.eigenvectors = torch.as_tensor(eigenvectors, dtype=torch.float32)
        self.eigenvalues = torch.as_tensor(eigenvalues, dtype=torch.float32)
        self.alpha_std = alpha_std
    
    def __call__(self, image):
        alpha = torch.randn(3, dtype=image.dtype, device=image.device) * self.alpha_std
        self.eigenvectors = self.eigenvectors.to(image.device)
        self.eigenvalues = self.eigenvalues.to(image.device)
        rgb_shift = eigenvectors @ (alpha * eigenvalues)
        rgb_shift = rgb_shift.view(3, 1, 1 )
        return image + rgb_shift
        
         


In [ ]:
class TinyImageNet(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = Path(root)
        self.split = split
        self.transform = transform

        # 200クラスのWordNet ID
        with open(self.root / "wnids.txt") as f:
            self.classes = [
                line.strip() for line in f if line.strip()
            ]

        self.class_to_idx = {
            cls: i for i, cls in enumerate(self.classes)
        }
        # words.txtを読み込む
        self.wnid_to_name = {}
        with open(self.root / "words.txt") as f:
            for line in f:
                parts = line.strip().split("\t", 1)
                if len(parts) == 2:
                    wnid = parts[0]
                    name = parts[1]

                    self.wnid_to_name[wnid] = name

        # cls index -> cls name
        self.class_name = {self.wnid_to_name.get(wnid, wnid) for wnid in self.classes}

        # samples
        self.samples = []

        #  -----------------
        #  Training data
        #  -----------------
        if split == "train":
            for cls in self.classes:
                image_dir = self.root / "train" / cls / "images"
                for image_path in image_dir.glob("*.JPEG"):
                    self.samples.append((image_path, self.class_to_idx[cls]))

        #  ------------------
        #  Validation data
        #  ------------------
        elif split == "val":
            annotation_file = self.root / "val" / "val_annotations.txt"
            image_dir = self.root / "val" / "images"
            with open(annotation_file) as f:
                for line in f:
                    parts = line.strip().split()
                    filename = parts[0]
                    cls = parts[1]
                    self.samples.append((image_dir / filename, self.class_to_idx[cls]))

        #  -------------------
        #  Test data
        #  -------------------
        elif split == "test":
            image_dir = self.root / "test" / "images"
            for image_path in sorted(image_dir.glob("*.JPEG")):
                self.samples.append((image_path, -1))

        else:
            raise ValueError("split must be 'train', 'val', or 'test'.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_wnid(self, class_idx):
        return self.classes[class_idx]
    
    def get_class_name(self, class_idx):
        return self.class_name[class_idx]


In [ ]:
stats_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ]
)
stats_dataset = TinyImageNet(root=DATA_ROOT, split="train", transform=stats_transform)
stats_loader = DataLoader(
    stats_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)
mean, covariance = compute_rgb_statistics(stats_loader)
print("mean: ")
print(mean)
print("covariance: ")
print(covariance)
eigenvalues, eigenvectors = torch.linalg.eigh(covariance)
print("eigenvalues: ")
print(eigenvalues)
print("eigenvectors: ")
print(eigenvectors)
mean_subtraction = MeanSubtraction(mean)
pca_augmentation = PCAColorAugmentation(eigenvectors=eigenvectors, eigenvalues=eigenvalues, alpha_std=0.1)


In [ ]:
train_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(256),
        v2.RandomCrop(224),
        v2.RandomHorizontalFlip(),
        v2.ToDtype(torch.float32, scale=True),
        pca_augmentation,
        mean_subtraction  
    ]
)

val_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(224),
        v2.ToDtype(torch.float32, scale=True),
        mean_subtraction
    ]
)

In [ ]:
train_dataset = TinyImageNet(root=DATA_ROOT, split="train", transform=train_transform)
val_dataset = TinyImageNet(root=DATA_ROOT, split="val", transform=val_transform)
test_dataset = TinyImageNet(root=DATA_ROOT, split="test", transform=val_transform)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128, # 要確認
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels[:10])

In [ ]:
class AlexNetLRN(nn.Module):
    def __init__(self, size=5, alpha=1e-4, beta=0.75, k=2.0):
        super().__init__()
        self.size = size
        self.alpha = alpha
        self.beta = beta
        self.k = k

    def forward(self, x):
        # x: [B, C, H, W]
        squared = x.pow(2)
        pad = self.size // 2
        squared = F.pad(squared, (0, 0, 0, 0, pad, pad))
        scale = torch.zeros_like(x)
        for i in range(self.size):
            scale += squared[:, i:i+x.size(1), :, :]
        scale = self.k + (self.alpha / self.size) * scale
        return x / scale.pow(self.beta)

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            AlexNetLRN(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            # Conv2
            nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2, groups=2),
            nn.ReLU(inplace=True),
            AlexNetLRN(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            # Conv3
            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            # Conv4
            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, stride=1, padding=1, groups=2),
            nn.ReLU(inplace=True),
            # Conv5
            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, stride=1, padding=1, groups=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        self.classifier = nn.Sequential(
            # FC6
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            # FC7
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            # FC8
            nn.Linear(4096, num_classes)
        )
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.constant_(module.bias, 0.0)
        nn.init.constant_(self.features[4].bias, 1.0)
        nn.init.constant_(self.features[10].bias, 1.0)
        nn.init.constant_(self.features[12].bias, 1.0)
        nn.init.constant_(self.classifier[1].bias, 1.0)
        nn.init.constant_(self.classifier[4].bias, 1.0)

In [ ]:
model = AlexNet(num_classes=200)
print(
    "conv1:",
    model.features[0].bias[0]
)

print(
    "conv2:",
    model.features[4].bias[0]
)

print(
    "conv3:",
    model.features[8].bias[0]
)

print(
    "conv4:",
    model.features[10].bias[0]
)

print(
    "conv5:",
    model.features[12].bias[0]
)

print(
    "fc6:",
    model.classifier[1].bias[0]
)

print(
    "fc7:",
    model.classifier[4].bias[0]
)

print(
    "fc8:",
    model.classifier[6].bias[0]
)

In [ ]:
checkpoint_dir = Path(PTH_ROOT)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_path = checkpoint_dir / "latest_path.pth"
best_path = checkpoint_dir / "best_path.pth"

# m1 mac用
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# cuda製GPU用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
criterion= nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=3)
start_epoch = 0
best_val_acc = 0.0

if latest_path.exists():
    print(f"load checkpoint: {latest_path}")
    checkpoint = torch.load(latest_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    print(f"Resume from epoch {start_epoch}")
    print(f"Best Val Acc" f"{best_val_acc:.4f}")
else:
    print("No checkpoint found. " "Start training from scratch.")



num_epochs = 90
for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(dim=1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_loss = running_loss / total
    train_acc = correct / total
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    print(f"Epoch [{epoch + 1 / num_epochs}] " f"Train Loss: {train_loss:.4f}" f"Train Acc: {train_acc:.4f}" f"Val Loss: {val_loss:.4f}" f"Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc
        }, best_path)
        print(f"Best model saved." f"{best_path}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc
    }, latest_path)
    print(f"Latest checkpoint saved." f"{latest_path}")

    



In [ ]:
test_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(224),
        v2.ToDtype(torch.float32, scale=True),
        mean_subtraction
    ]
)
test_datasset = TinyImageNet(root=DATA_ROOT, split="test", transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)


In [ ]:
model = AlexNet(num_classes=200).to(device)
checkpoint = torch.load("pth/best.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
predictions = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model.image(images)
        predicted = outputs.argmaz(dim=1)
        predictions.extend(predicted.cpu().tolist())

for i, class_idx in enumerate(predictions):
    image_path = test_dataset.sample[i]
    wnid = test_dataset.get_wnid(class_idx)
    class_name = test_dataset.get_class_name(class_idx)
    print(f"{image_path.name:20s}" f"-> {class_name}" f"({wnid})")


